# Inteligencia artificial con Python

Este notebook resuelve las tres actividades propuestas:
1. Creación y colaboración en Google Colab.
2. Implementación de algoritmos de *machine learning* con Iris.
3. Implementación de una CNN para clasificación de células con malaria.

## 1) Creación de un documento Google Colab

### A) Pasos para crear un documento en Google Colab
1. Acceder a [Google Colab](https://colab.research.google.com/?hl=es) con una cuenta de Google.
2. Pulsar **Archivo → Nuevo cuaderno**.
3. Renombrar el cuaderno (por ejemplo, `IA_Biomedica_Actividad.ipynb`).
4. Verificar que el entorno de ejecución sea Python (**Entorno de ejecución → Cambiar tipo de entorno de ejecución**).

### B) Pasos para compartir con todo el grupo
1. Pulsar botón **Compartir** (arriba a la derecha).
2. Añadir correos de los integrantes del grupo.
3. Asignar permisos adecuados (recomendado: **Editor** para trabajo colaborativo).
4. Opcionalmente, activar enlace compartido restringido al grupo.
5. Confirmar acceso pidiendo a cada integrante que abra y edite una celda de prueba.

## 2) Implementación de algoritmos de aprendizaje automático (Iris)

In [ ]:
# Librerías para la actividad de machine learning
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, precision_recall_fscore_support

In [ ]:
# Cargar dataset Iris desde seaborn
iris = sns.load_dataset('iris')
iris.head()

In [ ]:
# Separar variables predictoras (X) y etiquetas de clase (y)
X = iris.drop(columns=['species'])
y = iris['species']

# A) One Hot Encoding de las clases
encoder = OneHotEncoder(sparse_output=False)
y_one_hot = encoder.fit_transform(y.to_frame())

print('Clases detectadas:', list(encoder.categories_[0]))
print('Primeras 5 filas (One Hot):')
print(y_one_hot[:5])

In [ ]:
# B) División entrenamiento / test
# Se usa 80% entrenamiento y 20% test por ser un equilibrio habitual entre
# aprender suficientes patrones y reservar un bloque representativo para evaluación final.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Muestras entrenamiento: {len(X_train)} ({len(X_train)/len(X):.1%})')
print(f'Muestras test: {len(X_test)} ({len(X_test)/len(X):.1%})')

In [ ]:
# C) Entrenar y comparar SVM, Random Forest y Naive Bayes
models = {
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale'),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42),
    'Naive Bayes (Gaussian)': GaussianNB()
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='macro', zero_division=0
    )

    results.append({
        'Modelo': name,
        'Precision_macro': precision,
        'Recall_macro': recall,
        'F1_macro': f1
    })

results_df = pd.DataFrame(results).sort_values(by='F1_macro', ascending=False)
results_df

In [ ]:
# Visualizar reporte completo de clasificación por modelo
for name in models:
    print('=' * 70)
    print(name)
    print('=' * 70)
    y_pred = models[name].predict(X_test)
    print(classification_report(y_test, y_pred, zero_division=0))

### D) Justificación del mejor modelo
Tomando como criterio principal el **F1-score macro** (balancea precisión y recall entre clases),
el mejor modelo es el que aparece primero en la tabla `results_df`.

Si dos modelos quedan muy cercanos, la elección puede afinarse así:
- **SVM**: suele ser robusto en fronteras complejas y con pocos datos.
- **Random Forest**: más interpretable por importancia de variables y robusto a ruido.
- **Naive Bayes**: muy rápido y útil como baseline o en escenarios con pocos recursos.

## 3) Implementación de aprendizaje profundo (CNN) para malaria

In [ ]:
# Librerías para Deep Learning
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

> **Nota de dataset:** Ajusta `DATASET_DIR` al directorio donde tengas las carpetas
> `Parasitized` y `Uninfected` del dataset de malaria descargado desde el aula virtual.

In [ ]:
# Configuración
DATASET_DIR = Path('malaria')  # Cambiar según ubicación real del dataset
IMG_SIZE = (100, 100)
SEED = 42
BATCH_SIZE = 32
EPOCHS = 20

np.random.seed(SEED)
tf.random.set_seed(SEED)

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f'No se encontró el directorio {DATASET_DIR.resolve()}. '
        'Descarga/copia el dataset y actualiza DATASET_DIR.'
    )

In [ ]:
# A) Cargar y preprocesar imágenes con OpenCV
def load_images(base_dir, img_size=(100, 100)):
    X, y = [], []
    label_map = {'Parasitized': 1, 'Uninfected': 0}

    for class_name, label in label_map.items():
        class_dir = base_dir / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f'No existe la carpeta: {class_dir}')

        for img_name in os.listdir(class_dir):
            img_path = class_dir / img_name
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size)
            img = img.astype('float32') / 255.0

            X.append(img)
            y.append(label)

    return np.array(X, dtype='float32'), np.array(y, dtype='int32')

X_img, y_img = load_images(DATASET_DIR, IMG_SIZE)
print('Forma X:', X_img.shape)
print('Forma y:', y_img.shape)

In [ ]:
# División en entrenamiento, validación y prueba (70/15/15 aprox.)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_img, y_img, test_size=0.15, random_state=SEED, stratify=y_img
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.1765,
    random_state=SEED,
    stratify=y_trainval
)

print('Entrenamiento:', X_train.shape, y_train.shape)
print('Validación:', X_val.shape, y_val.shape)
print('Prueba:', X_test.shape, y_test.shape)

In [ ]:
# B) Diseñar CNN con TensorFlow/Keras
def build_cnn(input_shape=(100, 100, 3)):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),

        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),

        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )

    return model

model = build_cnn((IMG_SIZE[0], IMG_SIZE[1], 3))
model.summary()

**Justificación breve del diseño:**
- **Función de activación final:** `sigmoid`, porque la salida es binaria (infectada/no infectada).
- **Función de pérdida:** `binary_crossentropy`, estándar para clasificación binaria probabilística.
- **Canales de entrada:** **3 canales (RGB)**, porque las imágenes en color conservan información cromática útil para distinguir patrones celulares.

In [ ]:
# Entrenamiento
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# C) Gráficas de pérdida y precisión
hist = history.history

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(hist['loss'], label='train_loss')
plt.plot(hist['val_loss'], label='val_loss')
plt.title('Evolución de la pérdida')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist['accuracy'], label='train_acc')
plt.plot(hist['val_accuracy'], label='val_acc')
plt.title('Evolución de la precisión')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluación final en test
test_metrics = model.evaluate(X_test, y_test, verbose=0)
for metric_name, metric_value in zip(model.metrics_names, test_metrics):
    print(f'{metric_name}: {metric_value:.4f}')

y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Uninfected', 'Parasitized'])
cmd.plot(cmap='Blues')
plt.title('Matriz de confusión - CNN malaria')
plt.show()

In [ ]:
# Mostrar ejemplos correctos e incorrectos
def show_examples(images, y_true, y_pred, max_show=6):
    correct_idx = np.where(y_true == y_pred)[0][:max_show]
    wrong_idx = np.where(y_true != y_pred)[0][:max_show]

    def plot_group(indices, title):
        n = len(indices)
        if n == 0:
            print(f'No hay ejemplos para: {title}')
            return

        plt.figure(figsize=(3*n, 3))
        for i, idx in enumerate(indices, 1):
            plt.subplot(1, n, i)
            plt.imshow(images[idx])
            plt.axis('off')
            plt.title(f'T:{y_true[idx]} | P:{y_pred[idx]}')
        plt.suptitle(title)
        plt.tight_layout()
        plt.show()

    plot_group(correct_idx, 'Predicciones correctas')
    plot_group(wrong_idx, 'Predicciones incorrectas')

show_examples(X_test, y_test, y_pred)